In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sentence_transformers import SentenceTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from lightgbm import LGBMClassifier

In [2]:
import sys
sys.path.append("..")

In [3]:
books = pd.read_csv("../data/books_with_categories.csv")

print("Dataset shape:", books.shape)

books.head()

Dataset shape: (5197, 14)


,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description,simple_categories
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...,Fiction
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...,Fiction
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le...",Fiction


In [4]:
books["simple_categories"] = books["simple_categories"].replace({
    "Children's Fiction": "Fiction",
    "Children's Nonfiction": "Nonfiction"
})

In [5]:
books["simple_categories"].value_counts()

simple_categories
Fiction       3470
Nonfiction    1727
Name: count, dtype: int64

In [6]:
books = books[
    books["simple_categories"].isin(["Fiction", "Nonfiction"])
].copy()

texts = books["description"].fillna("").tolist()

labels = books["simple_categories"].map({
    "Fiction": 0,
    "Nonfiction": 1
}).values

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

In [8]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
# Embedding Cache System

train_cache = "../embeddings/train_embeddings.npy"
test_cache = "../embeddings.test_embeddings.npy"

In [10]:
if os.path.exists(train_cache):

    X_train_embeddings = np.load(train_cache)

else:

    X_train_embeddings = embedding_model.encode(
        X_train,
        batch_size=64,
        show_progress_bar=True
    )

    np.save(train_cache, X_train_embeddings)

Batches:   0%|          | 0/65 [00:00<?, ?it/s]

In [11]:
if os.path.exists(test_cache):

    X_test_embeddings = np.load(test_cache)

else:

    X_test_embeddings = embedding_model.encode(
        X_test,
        batch_size=64,
        show_progress_bar=True
    )

    np.save(test_cache, X_test_embeddings)

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

In [12]:
classifier = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=10
)

classifier.fit(X_train_embeddings, y_train)

[LightGBM] [Info] Number of positive: 1381, number of negative: 2776
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.036659 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 97920
[LightGBM] [Info] Number of data points in the train set: 4157, number of used features: 384
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.332211 -> initscore=-0.698203
[LightGBM] [Info] Start training from score -0.698203


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,10
,learning_rate,0.05
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [13]:
y_pred = classifier.predict(X_test_embeddings)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print(classification_report(y_test, y_pred))

Accuracy: 0.8028846153846154
              precision    recall  f1-score   support

           0       0.82      0.90      0.86       694
           1       0.76      0.60      0.67       346

    accuracy                           0.80      1040
   macro avg       0.79      0.75      0.76      1040
weighted avg       0.80      0.80      0.80      1040



F:\venv\Projects\book-recommendation-project\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [14]:
joblib.dump(classifier, "../models/book_category_classifier.pkl")

['../models/book_category_classifier.pkl']

In [15]:
classifier = joblib.load("../models/book_category_classifier.pkl")

In [16]:
label_map = {
    0: "Fiction",
    1: "Nonfiction"
}

def predict_book_category(text):

    embedding = embedding_model.encode([text])

    prediction = classifier.predict(embedding)[0]

    return label_map[prediction]

In [17]:
examples = [
    "A young wizard begins his magical training at a mysterious school",
    "The biography of Albert Einstein and his groundbreaking discoveries",
    "A thrilling mystery about a detective solving crimes in London"
]

for text in examples:

    print(text)
    print("Prediction:", predict_book_category(text))
    print()

A young wizard begins his magical training at a mysterious school
Prediction: Fiction

The biography of Albert Einstein and his groundbreaking discoveries
Prediction: Nonfiction

A thrilling mystery about a detective solving crimes in London
Prediction: Fiction



F:\venv\Projects\book-recommendation-project\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
F:\venv\Projects\book-recommendation-project\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
F:\venv\Projects\book-recommendation-project\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
